In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


# Live — Real-Time Image Viewer

`Live` is designed for streaming — each `.push()` sends only the new frame to the browser, not all previous frames. Toggle between 2D (filmstrip with newest on top) and 3D (stack with playback) modes at any time.

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import threading
import time
import numpy as np
import torch
from quantem.widget import Live, IO, profile
profile()

quantem.widget  0.0.14
quantem         0.1.7
Python          3.14.3 (CPython)
NumPy           2.4.2
PyTorch         2.10.0
Compute         mps (arm)
System RAM      24.0 GB (4.2 GB available)
VRAM            shared (unified memory)
Disk            148.3 GB free / 926.3 GB
Jupyter         Lab 4.5.6, notebook 7.5.5
anywidget       0.9.21
Platform        macOS 26.0.1 arm64


/Users/macbook/miniforge3/envs/bob-env/lib/python3.14/site-packages/anywidget/_util.py:283: UserWarning: anywidget: Live-reloading feature is disabled. To enable, please install the 'watchfiles' package.
  start_thread=_should_start_thread(path),


In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

def make_lattice(size=512, defocus=0.0, seed=42):
    """Synthetic HRTEM lattice with varying defocus."""
    rng = torch.Generator(device="cpu").manual_seed(seed)
    y = torch.linspace(-5, 5, size, device=device)
    x = torch.linspace(-5, 5, size, device=device)
    Y, X = torch.meshgrid(y, x, indexing="ij")
    a1 = torch.sin(2 * np.pi * (X + 0.5 * Y) * 2.5)
    a2 = torch.sin(2 * np.pi * Y * 2.5 * np.sqrt(3) / 2)
    lattice = (a1 + a2) ** 2
    sigma = 0.3 + abs(defocus) * 0.15
    blur = torch.exp(-(X**2 + Y**2) / (2 * sigma**2))
    image = lattice * (0.3 + 0.7 * blur)
    noise = torch.randn(size, size, generator=rng) * 0.05
    return (image.cpu() + noise).numpy().astype(np.float32)

## Push images one at a time

Each `.push()` sends only the new frame — O(1), not O(N). The filmstrip shows newest on top.

In [ ]:
w = Live(title="Focal Series", cmap="inferno")
w.push(make_lattice(512, defocus=-1.0, seed=0), label="Δf = -1.0")
w.push(make_lattice(512, defocus=-0.5, seed=1), label="Δf = -0.5")
w.push(make_lattice(512, defocus=0.0, seed=2), label="Δf = 0.0")
# Constructor batch is also supported:
# w = Live([img1, img2, img3], title="Focal Series", cmap="inferno")
print(w)
w

## Simulate live acquisition

Push new images on a timer — watch the filmstrip grow in real time. Toggle to 3D mode in the controls to scrub through frames as a stack.

In [5]:
def simulate_acquisition(widget, n=8, delay=1.5):
    for i in range(n):
        time.sleep(delay)
        df = -2.0 + i * 0.5
        widget.push(make_lattice(512, defocus=df, seed=100 + i), label=f"Δf = {df:+.1f}")
        print(f"  Pushed frame {widget.n_frames} (Δf = {df:+.1f})")
    print("Acquisition complete.")
acq = threading.Thread(target=simulate_acquisition, args=(w,), daemon=True)
acq.start()
print("Acquiring... watch the widget above!")

Acquiring... watch the widget above!


## 3D mode — start as a stack

Set `mode="3d"` to view incoming data as a playable stack. Hit play to animate through the time series.

In [ ]:
# Use constructor with a 3D stack for reliable batch loading
stack = np.stack([make_lattice(256, defocus=-1.0 + i * 0.5, seed=200 + i) for i in range(5)])
w3d = Live(stack, title="Drift Monitor", mode="3d", cmap="viridis", fps=3.0)
print(w3d)
w3d

## Watch a folder

`Live.watch()` monitors a directory and auto-pushes new files. Same as `IO.watch` but built into the widget.

In [7]:
import tempfile
watch_dir = tempfile.mkdtemp(prefix="live_watch_")
np.save(f"{watch_dir}/frame_000.npy", make_lattice(512, seed=300))
w_watch = Live.watch(watch_dir, pattern="*.npy", interval=1.0, cmap="magma")
w_watch

Live.watch: loaded 1 existing file(s), 1 frames


Live(1 frames, mode=2D, watching)

In [8]:
# Simulate files arriving (different sizes too)
def drop_files(folder, n=5, delay=2.0):
    sizes = [256, 512, 512, 1024, 512]
    for i in range(n):
        time.sleep(delay)
        sz = sizes[i % len(sizes)]
        np.save(f"{folder}/frame_{i+1:03d}.npy", make_lattice(sz, defocus=i * 0.3, seed=300 + i + 1))
        print(f"  Dropped frame_{i+1:03d}.npy ({sz}×{sz})")
    print("Done.")
drop_thread = threading.Thread(target=drop_files, args=(watch_dir,), daemon=True)
drop_thread.start()
print("Dropping files... watch the widget above!")

Dropping files... watch the widget above!


## Cleanup

In [9]:
w_watch.stop()
print("Watcher stopped.")

Watcher stopped.
  Pushed frame 4 (Δf = -2.0)
  Dropped frame_001.npy (256×256)
  Pushed frame 5 (Δf = -1.5)
  Dropped frame_002.npy (512×512)
  Pushed frame 6 (Δf = -1.0)
  Pushed frame 7 (Δf = -0.5)
  Dropped frame_003.npy (512×512)
  Pushed frame 8 (Δf = +0.0)
  Dropped frame_004.npy (1024×1024)
  Pushed frame 9 (Δf = +0.5)
  Dropped frame_005.npy (512×512)
Done.
  Pushed frame 10 (Δf = +1.0)
  Pushed frame 11 (Δf = +1.5)
Acquisition complete.
